# RAG Retrieval Accuracy — Facet-Text Embedding (per trait)

For each trait, embeds **only the 6 trait-relevant facet lines** of each essay's
profile through the embedding model — no raw post text, no fusion.

| | Query vector | Training index |
|---|---|---|
| **This notebook** | `embed(6_facet_slice)` | per-trait in-memory index |
| `profile` baseline | `embed(full_30_facet_profile)` | shared index |
| `sliced_dual_asym` | `norm(α·embed(raw) + (1-α)·embed(6_facets))` | shared dual index |

**Hypothesis:** isolating just the trait-relevant facet text removes cross-trait
noise from the embedding space and makes the 768-d vector more label-discriminative.

**One index per trait:** training essays are embedded separately for each of the
5 traits using that trait's 6-facet slice — so the retrieval space is tailored
to each trait rather than shared.

**Prerequisites:**
- `data/profile_db/essays/profile_store.jsonl` — training profiles
- `data/profile_db/essays_val/profile_store.jsonl` — val profiles (from `rag_retrieve_accuracy_profile.ipynb`)

In [1]:
import os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

project_root = Path.cwd().resolve()
if not (project_root / "ptd_model").exists():
    project_root = (project_root / ".." / "..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: F:\std\GR\code\model_x_ocean


## Configuration

In [ ]:
TRAIN_CSV        = str(project_root / "data/split/essays/train.csv")
VAL_CSV          = str(project_root / "data/split/essays/val.csv")
TRAIN_PROFILE_DB = str(project_root / "data/profile_db/essays/profile_store.jsonl")
VAL_PROFILE_DB   = str(project_root / "data/profile_db/essays_val/profile_store.jsonl")
RES_DIR          = str(project_root / "result/retrieve_accuracy/facet_embed")

K_VALUES    = [3, 5]
FACET_ALPHA = 0.8   # weight on facet-slice embedding
RAW_ALPHA   = 0.2   # weight on raw-post embedding

TRAITS = {
    "cOPN": "Openness to Experience",
    "cCON": "Conscientiousness",
    "cEXT": "Extraversion",
    "cAGR": "Agreeableness",
    "cNEU": "Neuroticism",
}

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
print(f"Train split : {len(train_df)} rows")
print(f"Val split   : {len(val_df)} rows")

for label, p in [
    ("Train profiles", TRAIN_PROFILE_DB),
    ("Val profiles",   VAL_PROFILE_DB),
]:
    status = "OK" if Path(p).exists() else "MISSING"
    print(f"  {label:<16s}: [{status}]  {p}")

if not Path(TRAIN_PROFILE_DB).exists():
    raise RuntimeError("Missing training profiles.")
if not Path(VAL_PROFILE_DB).exists():
    raise RuntimeError("Missing val profiles — run rag_retrieve_accuracy_profile.ipynb first.")

## Step 1 — Load training and val profiles

In [3]:
from rag.profiler.store   import ProfileStore
from rag.profiler.prompts import slice_profile_for_trait

# Training profiles — keyed by user_id string (e.g. "user_0")
train_store = ProfileStore(TRAIN_PROFILE_DB)
train_store.load()
train_profiles = {
    e["user_id"]: e
    for e in train_store.get_all() if e.get("valid")
}
print(f"Training profiles loaded : {len(train_profiles)} / {len(train_df)}")

# Val profiles — keyed by row index (int)
val_store = ProfileStore(VAL_PROFILE_DB)
val_store.load()
val_profiles = {
    int(e["user_id"].split("_")[1]): e
    for e in val_store.get_all() if e.get("valid")
}
print(f"Val profiles loaded      : {len(val_profiles)} / {len(val_df)}")

Training profiles loaded : 1974 / 1974
Val profiles loaded      : 247 / 247


## Step 2 — Helper: extract trait labels from a training row

In [4]:
def extract_trait_labels(row) -> dict:
    out = {}
    for col, name in TRAITS.items():
        v = str(row[col]).strip().lower()
        if v in ("1", "1.0", "high"):
            out[name] = "high"
        elif v in ("0", "0.0", "low"):
            out[name] = "low"
    return out


def normalize_label(val) -> str:
    s = str(val).strip().lower()
    if s in ("1", "1.0"): return "high"
    if s in ("0", "0.0"): return "low"
    return s

## Step 3 — Embed training essays per trait\n\nFor each training essay:\n- Embed the 6 trait-relevant facet lines with the embedding model\n- Embed the raw post text (computed once, reused across all traits)\n- Fuse: `normalise(0.8 · embed(facet_slice) + 0.2 · embed(raw_post))`\n\nFalls back to raw-only when a profile is missing.\nThis produces 5 separate fused embedding matrices — one per trait.

In [ ]:
from rag.embedder import get_embedding

def _fuse_vecs(facet_vecs: np.ndarray, raw_vecs: np.ndarray) -> np.ndarray:
    """normalise(FACET_ALPHA * facet + RAW_ALPHA * raw)"""
    fused = FACET_ALPHA * facet_vecs + RAW_ALPHA * raw_vecs
    norms = np.linalg.norm(fused, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return (fused / norms).astype("float32")


# train_meta[i] = {user_id, trait_labels} — shared across all traits
train_meta = []
for i, (_, row) in enumerate(train_df.iterrows()):
    uid = f"user_{i}"
    train_meta.append({
        "user_id":      uid,
        "trait_labels": extract_trait_labels(row),
    })

# Raw post embeddings — computed once, reused for all 5 traits
print(f"Embedding {len(train_df)} raw training posts ...")
t0 = time.time()
raw_train_texts = [str(row["text"]) for _, row in train_df.iterrows()]
raw_train_embs  = np.array(get_embedding(raw_train_texts), dtype="float32")
print(f"Done in {time.time()-t0:.1f}s  shape={raw_train_embs.shape}")

# Per-trait: embed 6-facet slice, fuse with raw
# train_embs[trait_code] -> np.ndarray (N_TRAIN, 768)
train_embs = {}

for trait_code, trait_full in TRAITS.items():
    facet_texts = []
    n_fallback  = 0
    for i, (_, row) in enumerate(train_df.iterrows()):
        uid     = f"user_{i}"
        profile = train_profiles.get(uid)
        sliced  = slice_profile_for_trait(profile, trait_code) if profile else ""
        if sliced.strip():
            facet_texts.append(sliced)
        else:
            facet_texts.append(str(row["text"]))
            n_fallback += 1

    print(f"\n[{trait_code}] Embedding {len(facet_texts)} facet texts  "
          f"(fallback to raw: {n_fallback}) ...")
    t0 = time.time()
    facet_vecs = np.array(get_embedding(facet_texts), dtype="float32")
    print(f"  Done in {time.time()-t0:.1f}s")

    train_embs[trait_code] = _fuse_vecs(facet_vecs, raw_train_embs)
    print(f"  Fused shape: {train_embs[trait_code].shape}  "
          f"(alpha_facet={FACET_ALPHA}, alpha_raw={RAW_ALPHA})")

print("\nAll training embeddings ready.")

## Step 4 — Embed val query essays per trait\n\nSame fusion: `normalise(0.8 · embed(facet_slice) + 0.2 · embed(raw_post))`.

In [ ]:
# Raw val post embeddings — computed once, reused for all 5 traits
print(f"Embedding {len(val_df)} raw val posts ...")
t0 = time.time()
raw_val_texts = [str(row["text"]) for _, row in val_df.iterrows()]
raw_val_embs  = np.array(get_embedding(raw_val_texts), dtype="float32")
print(f"Done in {time.time()-t0:.1f}s  shape={raw_val_embs.shape}")

# val_embs[trait_code] -> np.ndarray (N_VAL, 768)
val_embs = {}

for trait_code, trait_full in TRAITS.items():
    facet_texts = []
    n_fallback  = 0
    for i, (_, row) in enumerate(val_df.iterrows()):
        profile = val_profiles.get(i)
        sliced  = slice_profile_for_trait(profile, trait_code) if profile else ""
        if sliced.strip():
            facet_texts.append(sliced)
        else:
            facet_texts.append(str(row["text"]))
            n_fallback += 1

    print(f"\n[{trait_code}] Embedding {len(facet_texts)} val facet texts  "
          f"(fallback to raw: {n_fallback}) ...")
    t0 = time.time()
    facet_vecs = np.array(get_embedding(facet_texts), dtype="float32")
    print(f"  Done in {time.time()-t0:.1f}s")

    val_embs[trait_code] = _fuse_vecs(facet_vecs, raw_val_embs)

print("\nAll val embeddings ready.")

## Step 5 — Build one in-memory FAISS index per trait

In [7]:
# trait_indexes[trait_code] -> faiss.IndexFlatL2
trait_indexes = {}

for trait_code in TRAITS:
    vecs = train_embs[trait_code]
    dim  = vecs.shape[1]
    idx  = faiss.IndexFlatL2(dim)
    idx.add(vecs)
    trait_indexes[trait_code] = idx
    print(f"[{trait_code}] index: {idx.ntotal} vectors, dim={dim}")

[cOPN] index: 1974 vectors, dim=768
[cCON] index: 1974 vectors, dim=768
[cEXT] index: 1974 vectors, dim=768
[cAGR] index: 1974 vectors, dim=768
[cNEU] index: 1974 vectors, dim=768


## Step 6 — Evaluate retrieval accuracy

In [8]:
all_rows       = []
per_query_rows = []

for k in K_VALUES:
    print(f"\n{'='*60}")
    print(f"  k = {k}")
    print(f"{'='*60}")

    for trait_code, trait_full in TRAITS.items():
        index      = trait_indexes[trait_code]
        q_embs     = val_embs[trait_code]        # (N_VAL, 768)
        match_rates, hits = [], []

        for i, (_, row) in enumerate(val_df.iterrows()):
            true_label = normalize_label(row[trait_code])
            if true_label not in ("high", "low"):
                continue

            # Retrieve k*6 candidates, filter to those with a label for this trait
            q_vec = q_embs[i].reshape(1, -1)
            dists, idxs = index.search(q_vec, k * 6)
            dists, idxs = dists[0], idxs[0]

            filtered = []
            for idx in idxs:
                if idx < 0 or idx >= len(train_meta):
                    continue
                meta = train_meta[idx]
                if trait_full in meta["trait_labels"]:
                    filtered.append(meta)
                if len(filtered) >= k:
                    break

            n       = len(filtered)
            matches = sum(
                1 for r in filtered
                if r["trait_labels"][trait_full] == true_label
            )
            rate = matches / n if n > 0 else 0.0
            hit  = int(matches > 0)

            match_rates.append(rate)
            hits.append(hit)
            per_query_rows.append({
                "query_idx":   i,
                "trait":       trait_full,
                "k":           k,
                "true_label":  true_label,
                "n_retrieved": n,
                "match_count": matches,
                "match_rate":  rate,
                "hit":         hit,
            })

        mean_r = float(np.mean(match_rates))
        std_r  = float(np.std(match_rates))
        hit_r  = float(np.mean(hits))
        print(f"  {trait_full:<30s}  match={mean_r:.4f}+/-{std_r:.4f}  hit={hit_r:.4f}")

        all_rows.append({
            "trait":           trait_full,
            "k":               k,
            "n_queries":       len(match_rates),
            "mean_match_rate": mean_r,
            "std_match_rate":  std_r,
            "hit_rate":        hit_r,
        })

summary_df   = pd.DataFrame(all_rows)
per_query_df = pd.DataFrame(per_query_rows)
print("\nEvaluation complete.")


  k = 3
  Openness to Experience          match=0.5358+/-0.4129  hit=0.7045
  Conscientiousness               match=0.5830+/-0.3688  hit=0.8178
  Extraversion                    match=0.5304+/-0.3522  hit=0.8016
  Agreeableness                   match=0.5587+/-0.3888  hit=0.7733
  Neuroticism                     match=0.5614+/-0.3896  hit=0.7611

  k = 5
  Openness to Experience          match=0.5377+/-0.3994  hit=0.7692
  Conscientiousness               match=0.5814+/-0.3301  hit=0.8866
  Extraversion                    match=0.5393+/-0.3118  hit=0.8866
  Agreeableness                   match=0.5482+/-0.3553  hit=0.8462
  Neuroticism                     match=0.5676+/-0.3603  hit=0.8381

Evaluation complete.


## Step 7 — Display & compare against baselines

In [9]:
print("=== Mean match rate ===")
pivot = summary_df.pivot_table(index="trait", columns="k", values="mean_match_rate")
display(pivot.round(4))

print("\n=== Hit rate ===")
pivot_hit = summary_df.pivot_table(index="trait", columns="k", values="hit_rate")
display(pivot_hit.round(4))

=== Mean match rate ===


k,3,5
trait,,
Agreeableness,0.5587,0.5482
Conscientiousness,0.5830,0.5814
Extraversion,0.5304,0.5393
Neuroticism,0.5614,0.5676
Openness to Experience,0.5358,0.5377



=== Hit rate ===


k,3,5
trait,,
Agreeableness,0.7733,0.8462
Conscientiousness,0.8178,0.8866
Extraversion,0.8016,0.8866
Neuroticism,0.7611,0.8381
Openness to Experience,0.7045,0.7692


In [10]:
# Compare against previously saved baselines if available
baseline_paths = {
    "profile":    project_root / "result/retrieve_accuracy/profile/accuracy_summary.csv",
    "rawpost":    project_root / "result/retrieve_accuracy/rawpost/accuracy_summary.csv",
    "full_dual":  project_root / "result/retrieve_accuracy/sliced_dual/accuracy_summary.csv",
}

for k_val in K_VALUES:
    print(f"\n=== Match rate comparison  (k={k_val}) ===")
    this = summary_df[summary_df["k"] == k_val].set_index("trait")["mean_match_rate"]
    compare = pd.DataFrame({"facet_embed": this})

    for name, path in baseline_paths.items():
        if not path.exists():
            continue
        df = pd.read_csv(path)
        if "strategy" in df.columns:
            df = df[df["strategy"] == "full_dual"]
        sub = df[df["k"] == k_val].set_index("trait")["mean_match_rate"]
        compare[name] = sub

    compare["best_baseline"] = compare.drop(columns="facet_embed").max(axis=1)
    compare["delta_vs_best"] = compare["facet_embed"] - compare["best_baseline"]
    display(compare.round(4))


=== Match rate comparison  (k=3) ===


,facet_embed,profile,rawpost,full_dual,best_baseline,delta_vs_best
trait,,,,,,
Openness to Experience,0.5358,0.5236,0.5020,0.5169,0.5236,0.0121
Conscientiousness,0.5830,0.5209,0.5236,0.5155,0.5236,0.0594
Extraversion,0.5304,0.5209,0.5452,0.5209,0.5452,-0.0148
Agreeableness,0.5587,0.5466,0.5169,0.5358,0.5466,0.0121
Neuroticism,0.5614,0.5574,0.5439,0.5047,0.5574,0.0040



=== Match rate comparison  (k=5) ===


,facet_embed,profile,rawpost,full_dual,best_baseline,delta_vs_best
trait,,,,,,
Openness to Experience,0.5377,0.5368,0.5150,0.5174,0.5368,0.0008
Conscientiousness,0.5814,0.5012,0.5109,0.5304,0.5304,0.0510
Extraversion,0.5393,0.5336,0.5312,0.5142,0.5336,0.0057
Agreeableness,0.5482,0.5538,0.5198,0.5401,0.5538,-0.0057
Neuroticism,0.5676,0.5709,0.5320,0.5117,0.5709,-0.0032


## Step 8 — Save

In [11]:
os.makedirs(RES_DIR, exist_ok=True)

summary_path   = os.path.join(RES_DIR, "accuracy_summary.csv")
per_query_path = os.path.join(RES_DIR, "per_query_details.csv")

summary_df.to_csv(summary_path, index=False)
per_query_df.to_csv(per_query_path, index=False)

print(f"Saved summary   -> {summary_path}  ({len(summary_df)} rows)")
print(f"Saved per-query -> {per_query_path}  ({len(per_query_df)} rows)")

Saved summary   -> F:\std\GR\code\model_x_ocean\result\retrieve_accuracy\facet_embed\accuracy_summary.csv  (10 rows)
Saved per-query -> F:\std\GR\code\model_x_ocean\result\retrieve_accuracy\facet_embed\per_query_details.csv  (2470 rows)
